---

# Independent Soft Actor-Critic Agents with a Decomposed Action Space under Centralised Training and Centralised Execution (CTCE)

---

## Imports & Variables

In [1]:
import os
import sys
import ray
import glob
import numpy as np
from ray import tune
from ray.tune import ExperimentAnalysis
from agents.dual_sac_agent import DualPandaSACAgent
#from wrappers.multi_agent_mpe_wrapper import MPEMultiAgentWrapper
from wrappers.multi_agent_pandas_wrapper import DualPandaParallelEnv
from ray.rllib.env.wrappers.pettingzoo_env import ParallelPettingZooEnv
from utils.methods import plot_training_rewards, plot_per_agent_rewards, generate_snake_U_targets, force_print_to_terminal, plot_smooth_training_rewards

# Set the project directory to the current working directory
# and make sure it’s in the system path so local imports work
PROJECT_DIR = os.path.abspath(os.getcwd())
sys.path.append(PROJECT_DIR)
os.environ["PYTHONPATH"] = PROJECT_DIR

# Name of the custom environment
ENV_NAME = "dual-panda-env-v14"
# Path to the MuJoCo XML model file for the dual-arm cable simulation
XML_FILE_PATH = os.path.join(PROJECT_DIR, "models/two_robots_one_cable/dual_panda_cable_sim.xml")

# Function to create the environment instance
# Adds the project directory to sys.path just in case,
# then creates the DualPandaParallelEnv using the XML model file
def env_creator(config):
    sys.path.append(PROJECT_DIR)
    env = DualPandaParallelEnv(model_path=XML_FILE_PATH)
    return env

# Register the environment with Ray Tune using the given ENV_NAME
# This lets RLlib refer to it by name in the config
tune.register_env(ENV_NAME, lambda config: env_creator(config))

force_print_to_terminal()

<div style="height: 70px;"></div>

---

## Train

In [2]:
# Create an instance of the SAC training agent with the registered environments
agent = DualPandaSACAgent(env_name=ENV_NAME)

In [3]:
# Start training the agent and get the best checkpoint and training analysis results
best_checkpoint, analysis = agent.train()

2025-06-25 20:06:20,229	INFO worker.py:1752 -- Started a local Ray instance.
2025-06-25 20:06:22,428	INFO tune.py:613 -- [output] This uses the legacy output and progress reporter, as Jupyter notebooks are not supported by the new engine, yet. For more information, please see https://github.com/ray-project/ray/issues/36949
C:\Users\jacob\anaconda3\envs\petting_env\lib\site-packages\gymnasium\spaces\box.py:130: UserWarning: WARN: Box bound precision lowered by casting to float32
  gym.logger.warn(f"Box bound precision lowered by casting to {self.dtype}")
C:\Users\jacob\anaconda3\envs\petting_env\lib\site-packages\gymnasium\utils\passive_env_checker.py:164: UserWarning: WARN: The obs returned by the `reset()` method was expecting numpy array dtype to be float32, actual type: float64
  logger.warn(
C:\Users\jacob\anaconda3\envs\petting_env\lib\site-packages\gymnasium\utils\passive_env_checker.py:188: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation

(SAC pid=20940) 2025-06-25 20:06:29,842	WARNING deprecation.py:50 -- DeprecationWarning: `rllib/algorithms/simple_q/` has been deprecated. Use `rllib_contrib/simple_q/` instead. This will raise an error in the future!
(RolloutWorker pid=6280) 2025-06-25 20:06:43,141	WARNING env.py:298 -- Your MultiAgentEnv <DualPandaParallelEnv instance> does not have some or all of the needed base-class attributes! Make sure you call `super().__init__()` from within your MutiAgentEnv's constructor. This will raise an error in the future.
(SAC pid=20940) Trainable.setup took 15.936 seconds. If your trainable is slow to initialize, consider setting reuse_actors=True to reduce actor creation overheads.
(SAC pid=20940) Install gputil for GPU system monitoring.
(SAC pid=20940) 2025-06-25 20:06:47,884	WARNING multi_agent_prioritized_replay_buffer.py:215 -- Adding batches with column `weights` to this buffer while providing weights as a call argument to the add method results in the column being overwritten.

Trial name,agent_timesteps_total,connector_metrics,counters,custom_metrics,episode_len_mean,episode_media,episode_reward_max,episode_reward_mean,episode_reward_min,episodes_this_iter,info,num_agent_steps_sampled,num_agent_steps_trained,num_env_steps_sampled,num_env_steps_sampled_this_iter,num_env_steps_sampled_throughput_per_sec,num_env_steps_trained,num_env_steps_trained_this_iter,num_env_steps_trained_throughput_per_sec,num_faulty_episodes,num_healthy_workers,num_in_flight_async_reqs,num_remote_worker_restarts,num_steps_trained_this_iter,perf,policy_reward_max,policy_reward_mean,policy_reward_min,sampler_perf,sampler_results,timers
SAC_dual-panda-env-v14_7be5c_00000,9400,"{'ObsPreprocessorConnector_ms': 0.06700605154037476, 'StateBufferConnector_ms': 0.012506544589996338, 'ViewRequirementAgentConnector_ms': 0.2307415008544922}","{'num_env_steps_sampled': 4700, 'num_env_steps_trained': 2048, 'num_agent_steps_sampled': 9400, 'num_agent_steps_trained': 2048, 'last_target_update_ts': 4700, 'num_target_updates': 32}",{},500,{},-867.13,-880.964,-892.103,0,"{'learner': {'robot_0': {'learner_stats': {'allreduce_latency': 0.0, 'grad_gnorm': 3.609499931335449, 'actor_loss': -0.42231473326683044, 'critic_loss': 0.9937074780464172, 'alpha_loss': -0.11175211519002914, 'alpha_value': 0.96951383, 'log_alpha_value': -0.030960554, 'target_entropy': -4.0, 'policy_t': -0.1420620083808899, 'mean_q': 0.8682987689971924, 'max_q': 2.6249310970306396, 'min_q': -3.2675390243530273}, 'td_error': array([ 4.031354 , 2.2132702 , 0.31678128, 5.393217 , 4.8301067 , 1.1812162 , 0.433133 , 0.40240794, 3.7473128 , 1.8230687 , 0.24635303, 1.7379043 , 5.723881 , 0.7093185 , 1.8319769 , 3.1640792 , 0.31292415, 2.245627 , 0.4442472 , 5.9055476 , 0.90643597, 3.451292 , 4.474468 , 7.1989813 , 3.8610182 , 10.115158 , 2.2058747 , 1.0134194 , 4.5608478 , 0.33310747, 2.240334 , 5.1765103 ], dtype=float32), 'mean_td_error': 2.8822240829467773, 'model': {}, 'custom_metrics': {}, 'num_agent_steps_trained': 32.0, 'num_grad_updates_lifetime': 32.0, 'diff_num_grad_updates_vs_sampler_policy': 31.0}, 'robot_1': {'learner_stats': {'allreduce_latency': 0.0, 'grad_gnorm': 5.87604284286499, 'actor_loss': -2.7692017555236816, 'critic_loss': 0.10549677163362503, 'alpha_loss': -0.18289518356323242, 'alpha_value': 0.96935385, 'log_alpha_value': -0.03112557, 'target_entropy': -4.0, 'policy_t': 0.18444950878620148, 'mean_q': 0.9994032382965088, 'max_q': 2.121799945831299, 'min_q': -3.9589152336120605}, 'td_error': array([0.20723933, 0.52369785, 0.47501957, 3.3489013 , 0.9059263 , 0.08010542, 0.20277715, 0.5533613 , 0.39448923, 0.5698613 , 1.0952504 , 0.36506754, 0.67442787, 0.53307545, 0.640026 , 0.33067578, 1.3635134 , 0.13843602, 0.14964402, 0.16777098, 0.3959393 , 0.20839 , 0.23513818, 1.0162772 , 0.3518715 , 0.20770633, 3.6259968 , 0.3242582 , 0.38365287, 0.11079788, 0.4821396 , 0.89170647], dtype=float32), 'mean_td_error': 0.6547856330871582, 'model': {}, 'custom_metrics': {}, 'num_agent_steps_trained': 32.0, 'num_grad_updates_lifetime': 32.0, 'diff_num_grad_updates_vs_sampler_policy': 31.0}}, 'num_env_steps_sampled': 4700, 'num_env_steps_trained': 2048, 'num_agent_steps_sampled': 9400, 'num_agent_steps_trained': 2048, 'last_target_update_ts': 4700, 'num_target_updates': 32}",9400,2048,4700,200,128.497,2048,128,82.2383,0,4,0,0,128,"{'cpu_util_percent': 46.9, 'ram_util_percent': 92.23333333333333}","{'robot_0': -433.5650935399738, 'robot_1': -433.5650935399738}","{'robot_0': -440.4820499700636, 'robot_1': -440.4820499700636}","{'robot_0': -446.05174995287786, 'robot_1': -446.05174995287786}","{'mean_raw_obs_processing_ms': 1.013667231422111, 'mean_inference_ms': 4.370998367630502, 'mean_action_processing_ms': 0.556376116589287, 'mean_env_wait_ms': 15.771016603422773, 'mean_env_render_ms': 0.0}","{'episode_reward_max': -867.1301870799484, 'episode_reward_min': -892.1034999057536, 'episode_reward_mean': -880.9640999401267, 'episode_len_mean': 500.0, 'episode_media': {}, 'episodes_thi

(SAC pid=20940) 2025-06-25 20:06:59,611	WARNING deprecation.py:50 -- DeprecationWarning: `ray.rllib.execution.train_ops.multi_gpu_train_one_step` has been deprecated. This will raise an error in the future!
2025-06-25 20:07:22,523	INFO tune.py:1016 -- Wrote the latest version of all result files and experiment state to 'C:/Users/jacob/ray_results/SAC_2025-06-25_20-06-22' in 0.0451s.
(SAC pid=20940) Checkpoint successfully created at: Checkpoint(filesystem=local, path=C:/Users/jacob/ray_results/SAC_2025-06-25_20-06-22/SAC_dual-panda-env-v14_7be5c_00000_0_2025-06-25_20-06-22/checkpoint_000000)
2025-06-25 20:07:22,980	INFO tune.py:1048 -- Total run time: 60.55 seconds (59.82 seconds for the tuning loop).
2025-06-25 20:07:23,146	WARNING deprecation.py:50 -- DeprecationWarning: `rllib/algorithms/simple_q/` has been deprecated. Use `rllib_contrib/simple_q/` instead. This will raise an error in the future!
C:\Users\jacob\anaconda3\envs\petting_env\lib\site-packages\ray\rllib\algorithms\algori

<div style="height: 70px;"></div>

---

## Simulate

In [4]:
checkpoint = "C:/Users/jacob/ray_results/SAC_2025-06-25_20-06-22/SAC_dual-panda-env-v14_7be5c_00000_0_2025-06-25_20-06-22/checkpoint_000000"
agent.load_checkpoint(checkpoint)

env = env_creator({})

agent.simulate(env, num_episodes=5, render=True, test_threshold=0.05)

2025-06-25 20:04:46,864	INFO trainable.py:575 -- Restored on 127.0.0.1 from checkpoint: Checkpoint(filesystem=local, path=C:/Users/jacob/ray_results/SAC_2025-06-25_20-03-08/SAC_dual-panda-env-v14_07f7b_00000_0_2025-06-25_20-03-08/checkpoint_000000)


<div style="height: 70px;"></div>

---

## Plots

In [ ]:
progress_csv_path = "C:/Users/jacob/ray_results/SAC_2025-04-17_04-20-04/SAC_dual-panda-env-v14_dabcf_00000_0_2025-04-17_04-20-04/progress.csv"

In [ ]:
plot_smooth_training_rewards(progress_csv_path)

<div style="height: 70px;"></div>

---

## See Env

In [ ]:
import random
from wrappers.multi_agent_pandas_wrapper import DualPandaParallelEnv
from utils.methods import plot_training_rewards, plot_per_agent_rewards, generate_snake_U_targets, force_print_to_terminal

env = DualPandaParallelEnv(model_path="models/two_robots_one_cable/dual_panda_cable_sim.xml")
env.render()

#### Change Target Locations

In [ ]:
generate_snake_U_targets(env)
env.render()

<div style="height: 70px;"></div>

---

## MPE Testing

In [ ]:
from utils.methods import plot_smooth_training_rewards
progress_csv_path = "C:/Users/jacob/ray_results/SAC_2025-04-14_04-09-07/SAC_mpe-env_d463d_00000_0_2025-04-14_04-09-07/progress.csv"
plot_smooth_training_rewards(progress_csv_path)

In [ ]:
import os
import sys
from ray.tune.registry import register_env
from pettingzoo.mpe import simple_spread_v3, simple_adversary_v3, simple_push_v3, simple_tag_v3, simple_tag_v3
from agents.dual_sac_agent import DualPandaSACAgent
from wrappers.multi_agent_mpe_wrapper import MPEMultiAgentWrapper
from utils.methods import force_print_to_terminal, plot_multiple_smooth_training_rewards

force_print_to_terminal()

PROJECT_DIR = os.path.abspath(os.getcwd())
sys.path.append(PROJECT_DIR)
os.environ["PYTHONPATH"] = PROJECT_DIR

def mpe_env_creator(config):
    raw_env = simple_tag_v3.parallel_env(
        num_good=2,
        num_adversaries=2,
        num_obstacles=2,
        continuous_actions=True,
        max_cycles=25,
    )
    return MPEMultiAgentWrapper(raw_env)

register_env("mpe-env", mpe_env_creator)
env = mpe_env_creator({})

agent = DualPandaSACAgent(env_name="mpe-env", env_instance=env)
agent.train()

In [ ]:
from utils.methods import plot_multiple_training_rewards
plot_multiple_training_rewards(
    csv_paths=[
        "C:/Users/jacob/ray_results/SAC_2025-04-14_01-30-47_simple_spread_v3/SAC_mpe-env_b5f09_00000_0_2025-04-14_01-30-47/progress.csv",
        "C:/Users/jacob/ray_results/SAC_2025-04-14_02-24-05_simple_adversary_v3/SAC_mpe-env_27e1a_00000_0_2025-04-14_02-24-05/progress.csv",
        "C:/Users/jacob/ray_results/SAC_2025-04-14_03-18-10_simple_push_v3/SAC_mpe-env_b6587_00000_0_2025-04-14_03-18-11/progress.csv",
        "C:/Users/jacob/ray_results/SAC_2025-04-14_04-09-07_simple_tag_v3/SAC_mpe-env_d463d_00000_0_2025-04-14_04-09-07/progress.csv",
    ],
    labels=["simple_spread_v3", "simple_adversary_v3", "simple_push_v3", "simple_tag_v3"],
    window_size=50
)

### Simulate MPE Envs

In [ ]:
env.simulate(algo, episodes=3, render=True)

---

# References

Todorov, E., Erez, T., & Tassa, Y. (2012). MuJoCo: A physics engine for model-based control. In *2012 IEEE/RSJ International Conference on Intelligent Robots and Systems* (pp. 5026–5033). IEEE. https://doi.org/10.1109/IROS.2012.6386109

Zakka, K., Tassa, Y., & MuJoCo Menagerie Contributors. (2022). MuJoCo Menagerie: A collection of high-quality simulation models for MuJoCo. Retrieved from http://github.com/google-deepmind/mujoco_menagerie

Wu, Z., Liang, E., Luo, M., Mika, S., Gonzalez, J. E., & Stoica, I. (2021). RLlib Flow: Distributed Reinforcement Learning is a Dataflow Problem. In *Conference on Neural Information Processing Systems (NeurIPS)*. Retrieved from https://proceedings.neurips.cc/paper/2021/file/2bce32ed409f5ebcee2a7b417ad9beed-Paper.pdf

Liang, E., Liaw, R., Nishihara, R., Moritz, P., Fox, R., Goldberg, K., Gonzalez, J. E., Jordan, M. I., & Stoica, I. (2018). RLlib: Abstractions for Distributed Reinforcement Learning. In *International Conference on Machine Learning (ICML)*. Retrieved from https://arxiv.org/pdf/1712.09381

Brockman, G., Cheung, V., Pettersson, L., Schneider, J., Schulman, J., Tang, J., & Zaremba, W. (2016). OpenAI Gym. arXiv:1606.01540. Retrieved from https://arxiv.org/abs/1606.01540

Bou, A., Bettini, M., Dittert, S., Kumar, V., Sodhani, S., Yang, X., De Fabritiis, G., & Moens, V. (2023). TorchRL: A data-driven decision-making library for PyTorch. *arXiv*. Retrieved from https://arxiv.org/abs/2306.00577

Harris, C. R., Millman, K. J., van der Walt, S. J., Gommers, R., Virtanen, P., Cournapeau, D., Wieser, E., Taylor, J., Berg, S., Smith, N. J., Kern, R., Picus, M., Hoyer, S., van Kerkwijk, M. H., Brett, M., Haldane, A., Fernández del Río, J., Wiebe, M., Peterson, P., Gérard-Marchant, P., Sheppard, K., Reddy, T., Weckesser, W., Abbasi, H., Gohlke, C., & Oliphant, T. E. (2020). Array programming with NumPy. *Nature*, 585(7825), 357–362. https://doi.org/10.1038/s41586-020-2649-2

Terry, J., Black, B., Grammel, N., Jayakumar, M., Hari, A., Sullivan, R., Santos, L., Perez, R., Horsch, C., Dieffendahl, C., Williams, N., & Lokesh, Y. (2021). PettingZoo: Gym for multi-agent reinforcement learning. Retrieved from https://pettingzoo.farama.org/ and https://arxiv.org/abs/2009.14471

<div style="height: 70px;"></div>

---